# Day 16: Basic RAG System with Local Documentation

**RAG (Retrieval-Augmented Generation)** lets a language model answer questions using your own documents.

1. **Fetch** documentation pages and save them as text files.
2. **Split** the text into small chunks.
3. **Embed** each chunk (turn it into numbers) with `nomic-embed-text` and store it in **ChromaDB**.
4. **Retrieve** the chunks closest to your question.
5. **Generate** an answer with `qwen2.5:3b`, using only those chunks as context.

Everything runs locally through Ollama, so no paid APIs or cloud models are used.

### Setup and troubleshooting (Windows)
- Run this notebook with the Day 16 `.venv` kernel (VS Code: *Select Kernel* > the `.venv` interpreter).
- Ollama must be running (open the Ollama app or run `ollama serve` in a terminal).
- Models needed: `ollama pull nomic-embed-text` and `ollama pull qwen2.5:3b`.
- If a package is missing: `pip install requests beautifulsoup4 chromadb pandas`
- Some websites may block or time out. The notebook skips failed URLs and continues.
- Files are saved next to this notebook, so keep the notebook in your `Day 16` folder.

## Step 1: Import libraries and check that Ollama is running

In [1]:
import csv
import re
from pathlib import Path

import requests
import pandas as pd
import chromadb
from bs4 import BeautifulSoup

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen2.5:3b"

# Ask Ollama which models are installed
try:
    reply = requests.get(OLLAMA_URL + "/api/tags", timeout=10)
    reply.raise_for_status()
    installed = [m["name"] for m in reply.json()["models"]]
    print("Ollama is running. Installed models:", installed)
    for needed in [EMBED_MODEL, LLM_MODEL]:
        if not any(name.startswith(needed) for name in installed):
            print("WARNING: model not found ->", needed, "(run: ollama pull " + needed + ")")
except Exception as error:
    print("Cannot reach Ollama. Start it first, then re-run this cell.")
    print("Error:", error)

Ollama is running. Installed models: ['nomic-embed-text:latest', 'qwen2.5:3b']


## Step 2: Create folders and download the documentation
`Path` handles Windows backslashes for us. Each page is downloaded with `requests`, cleaned with BeautifulSoup, and saved as a `.txt` file.
Note: these URLs are documentation home pages, so each one contains a limited amount of text. Each page is capped at 30,000 characters to keep embedding fast.

In [2]:
# Folders are created next to this notebook
BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "documents"
OUTPUT_DIR = BASE_DIR / "outputs"
DB_DIR = BASE_DIR / "chroma_db"
for folder in [DOCS_DIR, OUTPUT_DIR, DB_DIR]:
    folder.mkdir(exist_ok=True)

URLS = {
    "python": "https://docs.python.org/3/",
    "huggingface": "https://huggingface.co/docs",
}
HEADERS = {"User-Agent": "Mozilla/5.0 (RAG learning project)"}
MAX_CHARS = 15000

for name, url in URLS.items():
    try:
        page = requests.get(url, headers=HEADERS, timeout=30)
        page.raise_for_status()
        soup = BeautifulSoup(page.text, "html.parser")
        # Remove parts that are not readable content
        for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        text = re.sub(r"\n\s*\n+", "\n", text).strip()[:MAX_CHARS]
        if len(text) < 100:
            print("SKIPPED (almost no text):", name)
            continue
        (DOCS_DIR / (name + ".txt")).write_text(text, encoding="utf-8")
        print("Saved:", name, "-", len(text), "characters")
    except Exception as error:
        print("FAILED:", name, "-", error)

print("Text files in documents folder:", [f.name for f in DOCS_DIR.glob("*.txt")])

Saved: python - 2758 characters
Saved: huggingface - 3011 characters
Text files in documents folder: ['huggingface.txt', 'python.txt']


## Step 3: Chunking, embeddings and ChromaDB
Helper functions that: split text into chunks (with a small overlap), embed text using Ollama, and store everything in a ChromaDB collection.
We build the main collection with **chunk size 400** characters.

In [3]:
def chunk_text(text, chunk_size, overlap=50):
    # Cut text into pieces of chunk_size characters; pieces overlap a little
    chunks = []
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks


def embed_texts(texts):
    # Ask Ollama for embeddings (a list of numbers for each text)
    reply = requests.post(OLLAMA_URL + "/api/embed",
                          json={"model": EMBED_MODEL, "input": texts}, timeout=300)
    reply.raise_for_status()
    return reply.json()["embeddings"]


chroma_client = chromadb.PersistentClient(path=str(DB_DIR))


def build_collection(chunk_size):
    # Build (or rebuild) a ChromaDB collection for the given chunk size
    name = "docs_chunk_" + str(chunk_size)
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass  # collection did not exist yet
    collection = chroma_client.create_collection(name, metadata={"hnsw:space": "cosine"})

    all_chunks, all_sources, all_ids = [], [], []
    for file in sorted(DOCS_DIR.glob("*.txt")):
        text = file.read_text(encoding="utf-8")
        for number, chunk in enumerate(chunk_text(text, chunk_size)):
            all_chunks.append(chunk)
            all_sources.append(file.name)
            all_ids.append(file.stem + "_" + str(number))

    # Embed and store in small batches
    for i in range(0, len(all_chunks), 32):
        batch = all_chunks[i:i + 32]
        collection.add(ids=all_ids[i:i + 32],
                       documents=batch,
                       embeddings=embed_texts(batch),
                       metadatas=[{"source": s} for s in all_sources[i:i + 32]])
    return collection


def retrieve(collection, question, top_k=3):
    # Find the top_k chunks closest to the question
    question_embedding = embed_texts([question])
    return collection.query(query_embeddings=question_embedding, n_results=top_k)


if len(list(DOCS_DIR.glob("*.txt"))) == 0:
    print("No documents found. Re-run Step 2 and check your internet connection.")
else:
    main_collection = build_collection(400)
    print("Chunks stored in ChromaDB (chunk size 400):", main_collection.count())

Chunks stored in ChromaDB (chunk size 400): 17


## Step 4: Ask a question (retrieval + answer from `qwen2.5:3b`)

In [4]:
question = input("Type your question about the documentation: ").strip()

if question == "":
    print("No question entered. Run this cell again.")
else:
    results = retrieve(main_collection, question, top_k=3)
    chunks = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]

    context = "\n\n".join(chunks)
    prompt = ("Answer the question using ONLY the context below. "
              "If the answer is not in the context, say you do not know.\n\n"
              "Context:\n" + context + "\n\nQuestion: " + question)

    reply = requests.post(OLLAMA_URL + "/api/chat",
                          json={"model": LLM_MODEL, "stream": False,
                                "messages": [{"role": "user", "content": prompt}]},
                          timeout=600)
    reply.raise_for_status()
    print("\nANSWER:\n" + reply.json()["message"]["content"])

    print("\nSOURCES USED:")
    for source, chunk in zip(sources, chunks):
        print("-", source, "|", chunk[:100].replace("\n", " "), "...")


ANSWER:
Python organizes its documentation to guide developers through a structured learning path, moving from basic syntax and features to more advanced topics and the standard library modules. The documentation is divided into sections such as "What's new in Python 3.15?" which helps developers understand new features in each version, and the "Tutorial" section which provides a tour of Python's syntax and features, starting from the basics.

For exploring built-in functions and classes, developers can refer to the "Built-ins reference" and "Built-in functions and classes" sections. This helps them understand the functions and objects already defined in the Python language itself.

To delve into advanced language features, developers can look into the "Language reference" and "For C/C++ programmers" sections. These sections provide a deep dive into the language's syntax, semantics, and more complex features.

For standard library modules, the "Library reference" and "Standard library

## Step 5: Compare chunk sizes (200, 400, 700)
Enter a question; retrieval is run for each chunk size and the top results are saved to a CSV file.
A lower **distance** means the chunk is closer in meaning to the question.
This step rebuilds three collections, so it can take a few minutes.

In [6]:
import ollama
compare_question = input("Type a question to compare chunk sizes: ").strip()

rows = []
answer_rows = []

if compare_question == "":
    print("No question entered. Run this cell again.")

else:
    for size in [200, 400, 700]:
        collection = build_collection(size)

        results = retrieve(collection, compare_question, top_k=3)

        chunks = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        # Combine retrieved chunks into context for answer generation
        context = "\n\n".join(
            f"Source: {metadatas[i]['source']}\n{chunks[i]}"
            for i in range(len(chunks))
        )

        prompt = f"""
You are a documentation assistant. Answer the question using
only the context provided below.

If the context does not contain enough information, clearly say
that the information is insufficient. Do not invent facts.

Context:
{context}

Question:
{compare_question}

Answer:
"""

        # Generate an answer using the local Ollama model
        response = ollama.chat(
            model="qwen2.5:3b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        answer = response["message"]["content"].strip()

        # Save the retrieved chunks and their distances
        for rank in range(len(chunks)):
            rows.append({
                "question": compare_question,
                "chunk_size": size,
                "total_chunks": collection.count(),
                "rank": rank + 1,
                "source": metadatas[rank]["source"],
                "distance": round(distances[rank], 4),
                "chunk_preview": chunks[rank][:150].replace("\n", " ")
            })

        # Save one generated response per chunk size
        answer_rows.append({
            "question": compare_question,
            "chunk_size": size,
            "retrieved_chunks": len(chunks),
            "answer": answer
        })

        print(f"\n{'=' * 65}")
        print(f"CHUNK SIZE: {size}")
        print(f"{'=' * 65}")
        print("\nGENERATED ANSWER:\n")
        print(answer)

        print("\nRETRIEVED SOURCES:")
        for metadata in metadatas:
            print("-", metadata["source"])

        print("\nRETRIEVAL DISTANCES:")
        print([round(distance, 4) for distance in distances])

        print("Completed chunk size:", size)

    # Save retrieval comparison
    comparison = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "chunk_size_comparison.csv"
    comparison.to_csv(csv_path, index=False, encoding="utf-8")

    # Save generated-answer comparison
    answer_comparison = pd.DataFrame(answer_rows)
    answer_csv_path = OUTPUT_DIR / "response_quality_comparison.csv"
    answer_comparison.to_csv(
        answer_csv_path,
        index=False,
        encoding="utf-8"
    )

    print("\nSaved retrieval comparison:", csv_path)
    print("Saved response comparison:", answer_csv_path)

    print("\nANSWER COMPARISON")
    display(answer_comparison)

    print("\nRETRIEVAL COMPARISON")
    display(comparison)

    # Refresh the main collection reference
    main_collection = chroma_client.get_collection(
        name="docs_chunk_400"
    )



CHUNK SIZE: 200

GENERATED ANSWER:

Python's documentation is organized in a way that guides developers from learning basic syntax and features to exploring advanced language features and standard-library modules. Here's a breakdown of how it organizes its documentation:

1. **Tutorial**: This section offers a tour of Python's syntax and features, which is essential for beginners to get started and understand the basic structure and functionalities of Python.

2. **Built-ins reference and Library reference**: These sections provide comprehensive documentation for Python’s built-in functions and classes, as well as the Standard Library modules. This is crucial for developers who are moving on to more advanced functionalities, as it covers the core libraries that come with Python installation.

3. **Language reference**: This section delves into the language elements and features, allowing developers to understand and utilize the more complex features of Python as they progress.

4. **W

,question,chunk_size,retrieved_chunks,answer
0,How does Python organize its documentation to ...,200,3,Python's documentation is organized in a way t...
1,How does Python organize its documentation to ...,400,3,Python organizes its documentation to guide de...
2,How does Python organize its documentation to ...,700,3,Python organizes its documentation to help dev...



RETRIEVAL COMPARISON


,question,chunk_size,total_chunks,rank,source,distance,chunk_preview
0,How does Python organize its documentation to ...,200,40,1,python.txt,0.2445,documents since Python 2.0 Tutorial Start here...
1,How does Python organize its documentation to ...,200,40,2,python.txt,0.2701,erence Standard library modules Language refer...
2,How does Python organize its documentation to ...,200,40,3,python.txt,0.2910,ocumentation Welcome! This is the official d...
3,How does Python organize its documentation to ...,400,17,1,python.txt,0.2489,tour of Python's syntax and features Built-ins...
4,How does Python organize its documentation to ...,400,17,2,python.txt,0.2718,3.15.1 Documentation Navigation index modules ...
5,How does Python organize its documentation to ...,400,17,3,python.txt,0.2815,ide Book List Audio/Visual Talks Python Develo...
6,How does Python organize its documentation to ...,700,10,1,python.txt,0.2580,3.15.1 Documentation Navigation index modules ...
7,How does Python organize its documentation to ...,700,10,2,python.txt,0.2927,"General index All functions, classes, and term..."
8,How does Python organize its documentation to ...,700,10,3,python.txt,0.3143,on 3.3 (EOL) Python 3.2 (EOL) Python 3.1 (EOL)...


## Step 6: Save a short Markdown summary of the experiment

In [7]:
if len(rows) == 0:
    print("Run Step 5 first so there are results to summarise.")
else:
    lines = ["# Chunk Size Experiment Summary", "",
             "**Question:** " + compare_question, "",
             "| Chunk size | Total chunks | Best distance | Average distance (top 3) | Top source |",
             "| --- | --- | --- | --- | --- |"]
    for size in [200, 400, 700]:
        part = comparison[comparison["chunk_size"] == size]
        best = part.iloc[0]
        lines.append("| " + str(size) + " | " + str(best["total_chunks"]) + " | "
                     + str(part["distance"].min()) + " | " + str(round(part["distance"].mean(), 4))
                     + " | " + best["source"] + " |")
    lines += ["", "Lower distance means a closer match. Distances come from different chunk sets, "
              "so read the differences as a rough guide, not proof that one size is best.",
              "", "Embedding model: " + EMBED_MODEL + ". Answer model: " + LLM_MODEL + "."]

    summary_path = OUTPUT_DIR / "experiment_summary.md"
    summary_path.write_text("\n".join(lines), encoding="utf-8")
    print("Saved:", summary_path)
    print("\n".join(lines))

Saved: e:\OneDrive - Indian Oil Corporation Limited\Desktop\AI-ML-Internship\Day 16\outputs\experiment_summary.md
# Chunk Size Experiment Summary

**Question:** How does Python organize its documentation to help developers move from learning basic syntax to exploring advanced language features and standard-library modules?

| Chunk size | Total chunks | Best distance | Average distance (top 3) | Top source |
| --- | --- | --- | --- | --- |
| 200 | 40 | 0.2445 | 0.2685 | python.txt |
| 400 | 17 | 0.2489 | 0.2674 | python.txt |
| 700 | 10 | 0.258 | 0.2883 | python.txt |

Lower distance means a closer match. Distances come from different chunk sets, so read the differences as a rough guide, not proof that one size is best.

Embedding model: nomic-embed-text. Answer model: qwen2.5:3b.
